# Three use cases with established numerical methods
### Voltage support, restoration, and cyber-attack response

This is the mathematical and experimental companion to the [three-hour workshop](Grid_Operator_Workflow.html). It provides the conventional tools that an agentic framework should call. Run each section during its corresponding use-case block, rather than adding another workshop.

**Choose your scope before you run.** Voltage support uses the actual supplied synthetic Greensboro feeder with either OpenDSS or coupled Python three-phase LinDistFlow. Restoration uses a small six-node graph so binary decisions and connectivity are inspectable. Cyber response uses an observable four-bus DC measurement/control example so WLS and stealth corruption are inspectable. The latter two examples do not represent full-feeder Greensboro optimization or state estimation.

The closing section shows actual stage responses from a separate fixed-versus-Llama run that invokes these numerical tools. The earlier 32-run workflow study remains unchanged. No result below is an assertion that Llama improves the underlying optimum.

**Prerequisites:** install your selected audience requirements, including SciPy. Running the numerical cells requires no language-model service. Running a new agentic trace explicitly requires local Ollama and Llama. Supplied stage traces are clearly labeled recorded observations.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import html
from src.standard_formulations import run_voltvar, solve_restoration_milp, run_wls_demo

ROOT = Path.cwd()
assert (ROOT / 'data/lindistflow3_network.json').exists(), 'Launch from the tutorial folder.'
BACKEND = 'lindistflow3'  # Change to 'opendss' in the full environment.
assert BACKEND in ('lindistflow3', 'opendss')
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
print('LIVE NUMERICAL EXPERIMENTS | selected voltage model:', BACKEND)
print('Restoration: six-node graph. Cyber: four-bus DC state-estimation/control example.')

LIVE NUMERICAL EXPERIMENTS | selected voltage model: lindistflow3
Restoration: six-node graph. Cyber: four-bus DC state-estimation/control example.


## General optimization contract

At each decision time, the numerical tool receives the estimated state, forecast, approved objectives and equipment permissions. For application $s$, prediction model $m$ and horizon $H$, write:

\[
\begin{aligned}
\min_{x_{0:H-1},\,u_{0:H-1},\,z_{0:H-1}}\quad
&\sum_{k=0}^{H-1}\ell_s(x_k,u_k,z_k;d_k)\\
\text{subject to}\quad
&h_m(x_k,u_k,z_k;d_k)=0,\\
&g_s(x_k,u_k,z_k;d_k)\leq0,\quad u_k\in\mathcal U_s,\quad z_k\in\{0,1\}^{n_z}.
\end{aligned}
\]

$x$ denotes electrical states, $u$ continuous controls, $z$ switching/pickup decisions, and $d$ supplied operating data. Power-flow equations are $h_m$; electrical limits, device capability and task restrictions are $g_s$. Add initialized state-transition equations, and any terminal cost or terminal constraint, when modeling a dynamic state such as battery energy. Set $H=1$ for a snapshot. Omit binary variables when equipment positions are fixed. The objective, units and weights are specified separately for each use case.

OpenDSS evaluates nonlinear unbalanced AC power flow. Python LinDistFlow supplies approximate coupled MV phase equations. **A power-flow simulator is not itself an optimizer:** a numerical optimization routine selects the controls and invokes or approximates the electrical model. A fresh model solve checks the candidate before review.

The LLM neither sets safety limits nor proves feasibility. A proposed reinterpretation of a priority or limit requires explicit readback. A fair fixed-versus-agentic comparison holds the numerical objective, model, information, checks and attempt budget constant. Otherwise a better optimizer could be mistaken for a benefit of the agent framework.


## Use case 1: Voltage support

### Established formulation: three-phase Volt/VAR OPF within MPC

The operator wants acceptable phase voltages and branch currents while retaining available PV production. The usual decision variables include inverter active power $p^g$ and reactive power $q^g$; taps and capacitors can be added when their actual capabilities and switching limits are available. A representative normalized objective is

\[
\min\sum_{k=0}^{H-1}\left[
\alpha_v\sum_{i,\phi}(v_{i\phi,k}-v^\star)^2
+\alpha_c\sum_g\frac{p^{av}_{g,k}-p^g_{g,k}}{S_g}
+\alpha_q\sum_g\left(\frac{q^g_{g,k}}{S_g}\right)^2
+\alpha_\Delta\|\widetilde u_k-\widetilde u_{k-1}\|_2^2\right].
\]

Here $v$ is voltage magnitude in per unit, $p,q,S$ share compatible power units, and $\widetilde u$ contains normalized controls. Positive weights express declared engineering priorities; there is no universal weight set. Constraints include three-phase power balance, $v_{min}\leq v\leq v_{max}$, $|I_\ell|\leq I^{max}_\ell$, $0\leq p^g\leq p^{av}$ and $(p^g)^2+(q^g)^2\leq S_g^2$, plus actual reactive and ramp limits. Only the first move of a feasible MPC plan is used before the next measurement update.

This is an established optimization family. The sources explain [optimal active/reactive inverter dispatch](https://research-hub.nlr.gov/en/publications/optimal-energy-dispatch-of-distributed-pvs-for-the-next-generatio-2/) and [multiphase OPF approximations](https://arxiv.org/abs/1406.3054). They motivate the formulation, rather than prescribing our teaching coefficients.



### Executed reduction: two continuous inverter participation controls

The numerical experiment sets $p_i=aS_i$ and $q_i=bS_i$ for every PV inverter. Available active power is set to rated kVA for this declared high-irradiance overlay. The two variables $a,b$ couple all devices; this is a restricted continuous OPF. Actual per-device kvar limits and a conservative 32-sided polygon inside $a^2+b^2\leq1$ limit capability.

Each iteration solves a local convex sensitivity QP with SciPy's SLSQP algorithm. The objective is $(1-a)^2+0.05b^2+10\,\mathrm{mean}[(v-1.01)^2]$. Local voltage/current sensitivities, a step bound and fresh backend validation control the iteration. The optimization uses small declared numerical margins; these are not a proof against uncertain disturbances. A local QP solution is not a global AC-optimality certificate.

The baseline has $a=1,b=0$, meaning no curtailment and no commanded reactive support. It isolates the effect of optimization. For attributing agent benefits, compare fixed and agentic supervisors using the **same** optimizer in the closing section. Local Volt/VAR droop is a distinct comparator that has not been evaluated here.

**Predict:** can reactive absorption improve voltage while increased current still requires active-power curtailment?


In [2]:
voltage = run_voltvar(BACKEND)
columns = ['p_actual_kw', 'q_actual_kvar', 'vmin', 'vmax',
           'max_line_utilization_pct', 'objective', 'accepted']
display(pd.DataFrame({name: {key: voltage[name][key] for key in columns}
                      for name in ('baseline', 'optimized')}).T)
display({'formulation': voltage['formulation'], 'validation': voltage['validation'],
         'violations_after': voltage['optimized']['violations'], 'scope': voltage['scope']})
recorded_methods = json.loads((ROOT / 'results/standard_formulations.json').read_text(encoding='utf-8'))
model_rows = []
for backend, record in recorded_methods['voltvar'].items():
    for kind in ('baseline', 'optimized', 'reference_full_ac_audit'):
        if kind not in record:
            continue
        point = record[kind]
        model_rows.append({'prediction_model': backend, 'evaluation': kind,
                           **{k: point[k] for k in ['vmin', 'vmax', 'max_line_utilization_pct', 'accepted']}})
print('Recorded model comparison and separate full-AC audit (not new local solves):')
display(pd.DataFrame(model_rows))

,p_actual_kw,q_actual_kvar,vmin,vmax,max_line_utilization_pct,objective,accepted
baseline,7850.446571,0.0,1.03,1.048024,60.82223,0.007605,True
optimized,7727.470994,-1248.591988,1.022481,1.037013,65.329358,0.005486,True


{'formulation': {'kind': 'continuous one-step P/Q participation OPF via local convex sensitivity QPs',
  'variables': 'a in [0,1], b in common inverter kvar/kVA bounds; p_i=a*S_i, q_i=b*S_i',
  'objective': '(1-a)^2 + 0.05*b^2 + 10*mean((v_pred-1.01)^2)',
  'inverter_capability': '32-side inscribed polygon within a^2+b^2<=1',
  'linear_constraints': '0.9502<=v_pred<=1.0498; I_pred/I_rating<=0.995',
  'physical_checks': '0.95<=v<=1.05; actual I/I_rating<=1; inverter circle',
  'solver': 'SciPy SLSQP solving quadratic objective with linear constraints',
  'finite_difference_step': 0.001,
  'trust_radius': 0.3,
  'available_p_kw': 7850.446570972884,
  'inverter_count': 930},
 'validation': {'fresh_backend_accepted': True, 'backend_solves': 28},
 'violations_after': {'voltage_low': 0,
  'voltage_high': 0,
  'line_current': 0,
  'inverter_capability': 0,
  'dispatch_tracking': 0},
 'scope': 'MV phases only'}

Recorded model comparison and separate full-AC audit (not new local solves):


,prediction_model,evaluation,vmin,vmax,max_line_utilization_pct,accepted
0,opendss,baseline,1.019639,1.108207,283.036633,False
1,opendss,optimized,1.004723,1.043224,99.891299,True
2,lindistflow3,baseline,1.030000,1.048024,60.822230,True
3,lindistflow3,optimized,1.022481,1.037013,65.329358,True
4,lindistflow3,reference_full_ac_audit,1.010868,1.087074,288.237096,False


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for key, color in [('baseline', '#c77630'), ('optimized', '#007f83')]:
    point = voltage[key]
    axes[0].plot(np.sort(point['voltages_pu']), color=color, label=key)
    axes[1].plot(np.sort(point['current_pu']) * 100, color=color, label=key)
for bound in (.95, 1.05): axes[0].axhline(bound, color='#a7353d', ls='--', lw=1)
axes[1].axhline(100, color='#a7353d', ls='--', lw=1)
axes[0].set(xlabel='Sorted phase sample', ylabel='Voltage (p.u.)', title=BACKEND + ': ' + voltage['scope'])
axes[1].set(xlabel='Sorted line sample', ylabel='Rated-current utilization (%)', title='Same-model current validation')
for ax in axes: ax.legend()
fig.tight_layout()
plt.show()

C:\Users\f00863x\AppData\Local\Temp\ipykernel_33912\262554514.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** Read current utilization together with voltage. Report whether the fresh selected-backend check accepted the candidate, then state the modeled scope. For LinDistFlow, a successful MV check says nothing about omitted LV constraints. Read optimizer status and the rejected-iteration trace before claiming convergence or optimality.

**Exercise, 3 minutes:** change `load_mult` to 0.7, rerun the numerical call and plot, and explain whether the binding constraint changed. Use the original finite-action MPC section of the main tutorial to study a horizon and the DQN proposal. This new continuous P/Q experiment is one-step optimization.

**Recorded comparison:** the linear proposal passes MV checks but fails the separate full-AC audit (maximum voltage about 1.087 p.u. and line utilization about 288.2%). Its baseline was already MV-feasible. The AC no-support baseline has a smaller unconstrained cost but violates limits; restoring feasibility can increase cost. The two objectives average voltage deviation over different model coverage, so neither their costs nor optimized controls isolate approximation error alone.


## Use case 2: Restoration

### Established formulation: mixed-integer service restoration

Let $y_i$ be binary load pickup, $z_\ell$ binary branch status, and $p_i^d$ nominal demand in kW. A common snapshot objective is

\[
\max\ \sum_i w_i p_i^d y_i-\lambda_{sw}\sum_\ell d_\ell,
\qquad d_\ell\geq z_\ell-z_\ell^0,\quad d_\ell\geq z_\ell^0-z_\ell.
\]

The weights $w_i$ encode approved service priorities. Require $d_\ell\geq0$ and $\lambda_{sw}>0$; the penalty has weighted-kW units per operation. A weighted sum does not guarantee strict critical-load-first service; use a lexicographic objective if that is the requirement. For a time sequence, integrate served power over interval durations and model pickup order explicitly.

Power balance, voltage/current limits, equipment availability and source connectivity constrain pickup. Faulted branches stay open. Commodity-flow connectivity plus an appropriate energized-edge count enforces a rooted radial network; an edge count alone is insufficient. With linearized power flow and polyhedral equipment limits the problem is a **MILP**. A conic formulation instead gives a **MISOCP**. Full nonlinear AC feasibility needs a separate check. See [Chen et al., sequential restoration](https://www.osti.gov/servlets/purl/1402471) and [Wang et al., radiality constraints](https://arxiv.org/abs/1912.05185).



### Executed reduction: graph MILP with independent verification

This six-node example chooses binary energized buses, picked-up loads and energized branches. Signed active-power flow obeys each branch's kW capacity. A fictitious commodity delivers one unit to every energized non-source bus, proving source connectivity; the edge count then enforces a tree. The isolated fault remains unavailable.

The first optimization maximizes priority-weighted served kW. A second optimization preserves that optimum and minimizes energized-branch count with a deterministic tie-break. This second objective is not a switching sequence or a count of changes from a pre-fault topology. An independent exhaustive enumeration checks the primary optimum on this small graph.

The example omits reactive flow, voltage drops and protection. The general formulation includes them; the existing Greensboro R1/R2 workflow checks its separately restricted configuration catalog in the selected electrical model.


In [4]:
restoration = {label: solve_restoration_milp(capacity)
               for label, capacity in [('normal', 220.), ('limited', 100.)]}
display(pd.DataFrame([{'case': label, 'tie_04_capacity_kw': next(edge['capacity_kw'] for edge in row['graph']['edges'] if edge['name'] == 'tie_04'),
                       **{k: row['milp'][k] for k in ['served_kw', 'critical_served_kw', 'objective', 'mip_gap']},
                       'enumeration_agrees': row['agreement']}
                      for label, row in restoration.items()]))
display(pd.DataFrame(restoration['limited']['graph']['edges']))
display({'load_blocks_kw': restoration['limited']['graph']['demand_kw'],
         'priorities': restoration['limited']['graph']['priority'],
         'limited_pickup': restoration['limited']['milp']['pickup'],
         'closed_edges': restoration['limited']['milp']['closed_edges']})

,case,tie_04_capacity_kw,served_kw,critical_served_kw,objective,mip_gap,enumeration_agrees
0,normal,220.0,250.0,90.0,1060.0,0.0,True
1,limited,100.0,200.0,90.0,1010.0,0.0,True


,name,u,v,capacity_kw,available
0,source_healthy,0,1,300.0,True
1,faulted_segment,1,2,250.0,False
2,section_23,2,3,250.0,True
3,section_34,3,4,250.0,True
4,section_25,2,5,250.0,True
5,tie_04,0,4,100.0,True
6,tie_15,1,5,100.0,True


{'load_blocks_kw': [0.0, 40.0, 60.0, 70.0, 50.0, 30.0],
 'priorities': [0.0, 1.0, 10.0, 1.0, 1.0, 10.0],
 'limited_pickup': [0, 1, 1, 1, 0, 1],
 'closed_edges': ['source_healthy',
  'section_34',
  'section_25',
  'tie_04',
  'tie_15']}

In [5]:
fig, ax = plt.subplots(figsize=(7, 3.5))
loads = np.array(restoration['normal']['graph']['demand_kw'])[1:]
positions = np.arange(len(loads))
for offset, (label, row) in zip([-.18, .18], restoration.items()):
    picked = np.array(row['milp']['pickup'])[1:]
    ax.bar(positions + offset, loads * picked, width=.36, label=label)
ax.set(xticks=positions, xticklabels=[str(i) for i in range(1, len(loads)+1)],
       xlabel='Load-block node', ylabel='Picked-up nominal load (kW)', title='MILP pickup under two capacity limits')
ax.legend()
fig.tight_layout()
plt.show()

C:\Users\f00863x\AppData\Local\Temp\ipykernel_33912\3237890838.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Exercise, 3 minutes:** reduce capacity to 80 kW or change the allowed graph in a separate copy. Identify which pickup changes and whether the solver still matches exhaustive enumeration. The critical-load weight is a declared preference, not a utility priority inferred by the LLM. Do not translate a final graph into physical switching orders without additional operational checks.


## Use case 3: Cyber-attack response

### Established foundation: state estimation and bad-data detection

For measurements $z=h(x)+a+e$, $e\sim\mathcal N(0,R)$ describes the assumed noise, and $a$ a possible corruption. The conventional WLS estimate is

\[
\widehat x=\arg\min_x(z-h(x))^\top R^{-1}(z-h(x)).
\]

For the full-rank linear teaching model $h(x)=Hx$ with known covariance, use residual $r=z-H\widehat x$, statistic $J=r^\top R^{-1}r$ and threshold $\chi^2_{m-n,1-\alpha}$, where $n$ counts identifiable free states after reference constraints. The normalized residual uses its own covariance $\Omega=R-H(H^\top R^{-1}H)^{-1}H^\top$: $r_i^N=|r_i|/\sqrt{\Omega_{ii}}$ when that variance is positive. Threshold calibration assumes the stated model and noise. An alarm indicates inconsistency, not proof of malicious intent. See the [PMDSE authors' bad-data documentation](https://electa-git.github.io/PowerModelsDistributionStateEstimation.jl/v0.4/bad_data/).

**Response is a second decision.** Quarantine only measurements justified by the response policy, restore observability from independently trusted measurements, estimate again, and call the voltage-support OPF with confirmed device permissions. If trusted data cannot support an observable estimate and a feasible plan, stop for operator review. Authentication, source provenance and permissions are separate controls; an LLM cannot manufacture them.

The key negative control is $a=Hc$: under the exact linear model, the estimate shifts by $c$ while the residual is unchanged. A residual detector can therefore miss a structured false-data injection. See [Liu, Ning and Reiter](https://reitermk.github.io/papers/2009/CCS1.pdf). This limitation is demonstrated in the notebook rather than hidden by testing only easy attacks.



### Executed reduction: observable DC WLS and constrained redispatch

The state consists of three non-reference bus angles in radians. The measurement model combines four line flows, three nodal injections and one angle, with explicitly stated independent Gaussian noise. Three scenarios share the same noise realization: nominal data, one corrupted flow measurement, and a structured $Hc$ shift. Inspect the known synthetic state error as well as the residual alarm.

An assumed protected set supplies three independent angle measurements for recovery. This restores observability but has zero residual redundancy, so no chi-square detection claim applies to that set. Trust is an explicit experimental assumption; the code does not authenticate sensors.

The recovery estimate actually enters a scalar convex DC redispatch problem: minimize half the square of a flexible injection, bound it between 0 and 0.5 p.u., and keep predicted line flows within the stated limits with a 0.01 p.u. engineering margin. The slack balances that injection. The known synthetic state provides a separate toy-model check. This demonstrates the estimator-to-controller link on the small DC network, not AC cyber resilience of the Greensboro feeder.


In [6]:
cyber = run_wls_demo()
display(pd.DataFrame([{'case': row['name'], 'state_error_rad': row['state_error_norm'],
                       **row['chi_square'], 'control_status': row['control']['status'],
                       'actual_toy_violations': row['control'].get('actual_violation_count')}
                      for row in cyber['cases']]))
display({'measurement_model': cyber['measurement_model'],
         'suspect_sensor_index': cyber['isolated_removal']['removed_index'],
         'trusted_recovery': cyber['trusted_recovery']})

,case,state_error_rad,statistic,degrees_of_freedom,alpha,threshold,alarm,control_status,actual_toy_violations
0,nominal_noise,0.000237,3.920222,5,0.01,15.086272,False,PROPOSE,0.0
1,isolated_corruption,0.005933,277.497447,5,0.01,15.086272,True,HOLD,NaN
2,stealth_Hc,0.026780,3.920222,5,0.01,15.086272,False,PROPOSE,1.0


{'measurement_model': {'H': [[-10.0, 0.0, 0.0],
   [8.0, -8.0, 0.0],
   [0.0, 10.0, -10.0],
   [0.0, 0.0, -5.0],
   [18.0, -8.0, 0.0],
   [-8.0, 18.0, -10.0],
   [0.0, -10.0, 15.0],
   [1.0, 0.0, 0.0]],
  'sigma': [0.004, 0.004, 0.004, 0.004, 0.006, 0.006, 0.006, 0.0005],
  'true_state': [-0.02, -0.05, -0.04],
  'state': 'three non-slack DC voltage angles in radians',
  'measurements': ['flow01',
   'flow12',
   'flow23',
   'flow03',
   'injection1',
   'injection2',
   'injection3',
   'angle1'],
  'noise_seed': 11,
  'attack_sensor_index': 1,
  'isolated_bias': 0.08,
  'stealth_shift': [0.01, 0.025, 0.0],
  'stealth_attack': [-0.1,
   -0.12000000000000001,
   0.25,
   0.0,
   -0.020000000000000007,
   0.37,
   -0.25,
   0.01]},
 'suspect_sensor_index': 1,
 'trusted_recovery': {'protected_H': [[1.0, 0.0, 0.0],
   [0.0, 1.0, 0.0],
   [0.0, 0.0, 1.0]],
  'protected_measurements': [-0.019982061329599805,
   -0.04992446613459283,
   -0.04008931656186556],
  'protected_sigma': [5e-05, 5e-

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
labels = ['Noise', 'Isolated corruption', 'Stealth Hc']
axes[0].bar(labels, [row['chi_square']['statistic'] for row in cyber['cases']], color=['#007f83', '#c77630', '#a7353d'])
axes[0].axhline(cyber['cases'][0]['chi_square']['threshold'], color='black', ls='--', label='1% false-alarm threshold')
axes[0].set(ylabel='Weighted residual statistic', title='Detection can miss structured corruption')
axes[0].tick_params(axis='x', rotation=12)
axes[0].legend()
recovered = cyber['trusted_recovery']['control']
axes[1].bar(np.arange(4), np.abs(recovered['actual_toy_model_flows_pu']), color='#007f83', label='After trusted recovery')
axes[1].plot(np.arange(4), recovered['line_limits_pu'], 'o--', color='#a7353d', label='Line limit')
axes[1].set(xticks=np.arange(4), xticklabels=['01', '12', '23', '03'], xlabel='DC model branch', ylabel='Absolute active flow (p.u.)', title='Trusted estimate enters constrained redispatch')
axes[1].legend()
fig.tight_layout()
plt.show()

C:\Users\f00863x\AppData\Local\Temp\ipykernel_33912\3932624953.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Exercise, 3 minutes:** compare the nominal and stealth residual statistics. Explain how both can pass while their state errors and resulting control decisions differ. Then remove one protected angle measurement in a copy of the experiment: an unobservable retained set must require review. A scalar margin is not a general uncertainty guarantee. These individual fixtures do not estimate an operational attack-detection rate.


## The same methods inside the agent workflow

The separate stage demonstration invokes these numerical tools for four specifications: voltage support with each electrical backend, limited-capacity restoration, and trusted DC recovery. Both fixed and agentic supervisors receive the same structured task and numerical parameters. Llama selects the tool family and gives a separately labeled explanation. An incompatible selection or failed numerical check requires review. Each run permits one numerical attempt; it does not claim recovery from arbitrary tool failures.

The table and traces below are **recorded actual experiments** in `results/standard_agent_workflow.json`. Reading this cell makes no model calls. The run journal preserves exact prompts, schema, responses and model identity. Select any agentic row to inspect all eight stages. The general production-style framework and the current reduced numerical scope must remain distinct in the interpretation.


In [8]:
standard_stages = json.loads((ROOT / 'results/standard_agent_workflow.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'case': row['case'], 'model': row['backend'], 'supervisor': row['supervisor'],
                       'status': row['status'], 'seconds': round(row['seconds'], 3),
                       'objective': (row['summary']['objective'] if 'objective' in row['summary']
                                     else row['summary'].get('control', {}).get('objective'))}
                      for row in standard_stages['runs']]))
print('Recorded actual Llama calls:', standard_stages['metadata']['actual_llama_calls'])
example = next(row for row in standard_stages['runs'] if row['case'] == 'restoration' and row['supervisor'] == 'agentic')
display(pd.DataFrame([{k: step[k] for k in ['stage', 'method', 'status']} for step in example['trace']]))
for step in example['trace']:
    exact = html.escape(json.dumps(step['response'], indent=2, ensure_ascii=False))
    label = html.escape(step['stage'] + ' — ' + step['status'])
    display(HTML('<details><summary>' + label + ': inspect the exact response</summary><pre>' + exact + '</pre></details>'))

,case,model,supervisor,status,seconds,objective
0,voltage,opendss,fixed,MODEL_RECOMMEND,6.845,0.397513
1,voltage,opendss,agentic,MODEL_RECOMMEND,12.490,0.397513
2,voltage,lindistflow3,fixed,MODEL_RECOMMEND,1.437,0.005486
3,voltage,lindistflow3,agentic,MODEL_RECOMMEND,7.384,0.005486
4,restoration,separate_reduced_teaching_model,fixed,MODEL_RECOMMEND,0.010,1010.000000
5,restoration,separate_reduced_teaching_model,agentic,REVIEW,6.102,1010.000000
6,cyber,separate_reduced_teaching_model,fixed,MODEL_RECOMMEND,0.002,0.007405
7,cyber,separate_reduced_teaching_model,agentic,REVIEW,5.935,0.007405


Recorded actual Llama calls: 8


,stage,method,status
0,Awareness,Declared scenario/schema and model-coverage ch...,PASS
1,Planning,RESTORATION_MILP with fixed parameters,SOLVED
2,Execution proposal,Extract the numerical solution; no physical ac...,PROPOSED
3,Simulation,Independent graph enumeration,EVALUATED
4,Verification,Declared numerical success and modeled-constra...,PASS
5,Recovery,One numerical attempt; unsuccessful verificati...,NOT_NEEDED
6,Explanation,Numerical metric card and separately labeled e...,REVIEW
7,Operator acknowledgment,Human review of immutable result identity,PENDING


**Read the stopping decision separately from feasibility.** In the supplied run, the numerical restoration and trusted-DC recovery checks passed, but the explanation stage selected `REVIEW`. That wording is broad in the model's choice schema. The wrapper treats it as a request to withhold a recommendation; it is not evidence that the numerical plan is infeasible. The response and final status are preserved. This experiment demonstrates integration and exposes an interface limitation, rather than claiming an agentic performance advantage.

### Optional new live stage run

Set `RUN_LIVE_AGENT=True` to make new local Llama calls and invoke the selected numerical tool again. Leave it false to finish the numerical workshop without another call. The saved display above remains a recorded trace. The local operator-review dashboard illustrates acknowledgment for the separately frozen feeder-workflow records; it does not acknowledge the new result automatically.


In [9]:
RUN_LIVE_AGENT = False
if RUN_LIVE_AGENT:
    from datetime import datetime, timezone
    from src.standard_agent_workflow import MethodLlama, run_case
    journal = ROOT / 'results' / ('classroom_standard_llama_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '.json')
    live = run_case('voltage', BACKEND, 'agentic', MethodLlama(journal))
    display(pd.DataFrame(live['trace']))
else:
    print('No new Llama calls requested. Numerical cells above ran live; the stage table is recorded.')

No new Llama calls requested. Numerical cells above ran live; the stage table is recorded.


## Scientific interpretation and references

The method experiments establish numerical behavior within their declared reductions. The matched stage runs show which tools the workflow actually invoked and whether their checks passed. They do not establish a better optimum, reduced operator workload, calibrated population detection rates, or field safety. Compare the existing authored-request experiment separately when discussing language-interface coverage.

Use [the formulation appendix](appendix/General_Optimization_Formulations.md) for full equations, constraints, units and primary references. The [main tutorial](Grid_Operator_Workflow.html) connects the three use cases to MPC, DQN, conventional comparisons, actual Llama errors and [operator acknowledgment](Operator_Dashboard.html).

Numerical implementation: `src/standard_formulations.py`. Stage integration: `src/standard_agent_workflow.py`. Baseline and new comparison records are separate from the earlier frozen studies. To regenerate all conventional-method records, run `python -m src.standard_formulations`. To explicitly repeat both-supervisor stage runs, run `python -m src.standard_agent_workflow` with both electrical dependencies and Ollama available. New stage runs keep separate journal directories.
